In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd


### Make manifest for 1-distractor anechoic tests

In [2]:
room_ir_df = pd.read_pickle('/om2/user/msaddler/spatial_audio_pipeline/assets/brir/mit_bldg46room1004/manifest_brir.pdpkl')
elevations = room_ir_df[(room_ir_df.src_dist == 1.4) & (room_ir_df.index_room == 0)].src_elev.unique() # np.arange(-20, 41, 10)
print(elevations)


[-20 -10   0  10  20  30  40]


## Dev front-back manifest 

In [3]:
dist_azimuths = np.arange(180, 271, 10) 
dist_azimuths = np.hstack([(360 - dist_azimuths)[1:][::-1], dist_azimuths])
dist_azimuths

array([ 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210,
       220, 230, 240, 250, 260, 270])

In [4]:
target_azims = np.arange(0, 91, 10)
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])
target_azims

array([  0,  10,  20,  30,  40,  50,  60,  70,  80,  90, 350, 340, 330,
       320, 310, 300, 290, 280, 270])

In [5]:
# First add conditions for confusion matrix 
target_azims = np.arange(0, 91, 10)
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])


dist_azimuths = np.arange(180, 271, 10) 
dist_azimuths = np.hstack([(360 - dist_azimuths)[1:][::-1], dist_azimuths])

azim_cond_dict = {ix:[(target_az, 0), (dist_az, 0)] for ix, (target_az, dist_az) in
              enumerate(itertools.product(*[target_azims, dist_azimuths]))} 

n_0_elev_conds = len(azim_cond_dict)
print(n_0_elev_conds)
# Now add elevation conds 
azims = np.array([0, 30, 90])
target_elevs = [-20, 10, 40]
distractor_elevs = [-20, -10,   0,  10,  20, 30,  40]
azims = np.concatenate([azims, 360 - azims[1:]])

# write function to reflect azimuths along midline. 270 is right 90 is left, 0 is front 180 is back

flip_back = lambda x: 180 - x if x < 180 else 540 - x

elev_cond_dict = {ix + n_0_elev_conds:[(azim, target_elev), (flip_back(azim), dist_elev)] for ix, (target_elev, dist_elev, azim) in
              enumerate(itertools.product(*[target_elevs, distractor_elevs, azims]))} 
print(len(elev_cond_dict))

# combine dicts 
all_cond_dict = {**azim_cond_dict, **elev_cond_dict}
print(len(all_cond_dict))

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

361
105
466


In [21]:
with open('binaural_test_manifests/front_back_conditions.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

In [22]:
with open("binaural_test_manifests/match_human_pilot_conds.pkl", 'rb') as f:
    match_human_pilot_conds = pickle.load(f)

## Make alt elevation conditions 


In [13]:
# First add conditions for confusion matrix 
target_azims = np.arange(0, 91, 10)

elevations = [-20, 40]
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])
azim_cond_dict = {ix:[(target_az, elev), (dist_az, elev)] for ix, (target_az, dist_az, elev) in
              enumerate(itertools.product(*[target_azims, target_azims, elevations]))} 

n_conds = len(azim_cond_dict)
print(n_conds)

722


In [15]:
# all_conds
# all conditions 
with open('binaural_test_manifests/alternate_elev_conditions.pkl', 'wb') as f:
    pickle.dump(azim_cond_dict, f)

## Match human pilot conditions

In [48]:
# First add conditions for confusion matrix 
target_azims = np.arange(0, 91, 10)
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])
azim_cond_dict = {ix:[(target_az, 0), (dist_az, 0)] for ix, (target_az, dist_az) in
              enumerate(itertools.product(*[target_azims, azimuths]))} 

n_0_elev_conds = len(azim_cond_dict)
print(n_0_elev_conds)
# Now add elevation conds 
azims = np.array([0, 10, 30, 90])
target_elevs = [-20, 10, 40]
distractor_elevs = [-20, -10,   0,  10,  20, 30,  40]
azims = np.concatenate([azims, 360 - azims[1:]])


elev_cond_dict = {ix + n_0_elev_conds:[(azim, target_elev), (azim, dist_elev)] for ix, (target_elev, dist_elev, azim) in
              enumerate(itertools.product(*[target_elevs, distractor_elevs, azims]))} 
print(len(elev_cond_dict))

# combine dicts 
all_cond_dict = {**azim_cond_dict, **elev_cond_dict}
print(len(all_cond_dict))

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

361
147
508


In [49]:
# all_conds
# all conditions 
with open('match_human_pilot_conds.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

### Test manifest combining conditions used in 11/29/2023 test with all conditions for 0 elev confusion matrix 

In [46]:
# First add conditions for confusion matrix 
target_azims = np.arange(0, 91, 10)
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])
cond_dict = {ix:[(target_az, 0), (dist_az, 0)] for ix, (target_az, dist_az) in
              enumerate(itertools.product(*[target_azims, azimuths]))} 

n_0_elev_conds = len(cond_dict)
print(n_0_elev_conds)
# Now add conditions for all other analyses 
target_azims = np.array([0, 10, 20, 30, 40, 60, 90])
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])


all_other_cond_dict = {ix + n_0_elev_conds:[(target_az, target_elev), (dist_az, dist_elev)] for ix, (target_az, target_elev, dist_az, dist_elev) in
              enumerate(itertools.product(*[target_azims, wanted_elevations, azimuths, wanted_elevations]))} 
print(len(all_other_cond_dict))

# combine dicts 
all_cond_dict = {**cond_dict, **all_other_cond_dict}
print(len(all_cond_dict))

# Assert keys contiguous
assert np.all(np.array(list(all_cond_dict.keys())) == np.arange(len(all_cond_dict)))

361
147
508


In [29]:
# all_conds
# all conditions 
with open('all_9253_room_conds_12_13_2023.pkl', 'wb') as f:
    pickle.dump(all_cond_dict, f)

## Add conditions for full confusion matrix at 0 elevation 

In [4]:
azimuths

array([  0,  10,  20,  30,  40,  50,  60,  70,  80,  90, 350, 340, 330,
       320, 310, 300, 290, 280, 270])

In [5]:
# target_azims = np.array([0, 20, 30, 40, 60, 90])
target_azims = np.array([10, 50, 70, 80])
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])

cond_dict = {ix:[(target_az, 0), (dist_az, 0)] for ix, (target_az, dist_az) in
              enumerate(itertools.product(*[target_azims, azimuths]))} 
len(cond_dict)


133

In [6]:
# all_conds
# all conditions 
with open('compliment_11_29_2023_locs_for_0_elev.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)

### For tests run 11/29/2023 and after

In [ ]:
target_azims = np.array([0, 20, 30, 40, 60, 90])
target_azims = np.concatenate([target_azims, 360 - target_azims[1:]])

cond_dict = {ix:[(target_az, target_elev), (dist_az, dist_elev)] for ix,(target_az, target_elev, dist_az, dist_elev) in
              enumerate(itertools.product(*[target_azims, wanted_elevations, azimuths, wanted_elevations]))} 
len(cond_dict)


In [25]:
# all_conds
# all conditions 
with open('expanded_all_azim_chosen_elev_az_11_29_2023.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)

#### For test ran approx 11/14/2023

In [64]:
all_pairs = []
for target_az, dist_az in itertools.product(*[azimuths, azimuths]): 
    all_pairs.append([(target_az, 60), (dist_az, 0)])
    all_pairs.append([(target_az, 0), (dist_az, 60)])
    all_pairs.append([(target_az, 40), (dist_az, -10)])
    all_pairs.append([(target_az, -10), (dist_az, 40)])

all_pairs_dict = {i: pair for i, pair in enumerate(all_pairs)}

In [66]:
len(all_pairs_dict)

1444

In [69]:
all_pairs_dict[1443]

[(270, -10), (270, 40)]

In [67]:
# all_conds
# all conditions 
with open('expanded_elev_v01_11_14_2023.pkl', 'wb') as f:
    pickle.dump(all_pairs_dict, f)

#### For test ran approx 10/29/2023

In [67]:

loc_conds_t_40_az_d_0_az = {ix:[(target_az, 40), (dist_az, 0)] for ix, (target_az, dist_az) in enumerate(itertools.product(*[azimuths, azimuths]))}
loc_conds_t_0_az_d_40_az = {ix:[(target_az, 0), (dist_az, 40)] for ix, (target_az, dist_az) in enumerate(itertools.product(*[azimuths, azimuths]))}


In [68]:
# write condition dicts to pickle files
with open('loc_conds_t_40_az_d_0_az.pkl', 'wb') as f:
    pickle.dump(loc_conds_t_40_az_d_0_az, f)

with open('loc_conds_t_0_az_d_40_az.pkl', 'wb') as f:
    pickle.dump(loc_conds_t_0_az_d_40_az, f)

In [84]:
# new dictionary with values of both dicts with keys indexed from 0 to 720
all_conds = {}
for i in range(722):
    if i <= 360:
        all_conds[i] = loc_conds_t_40_az_d_0_az[i]
    else:
        all_conds[i] = loc_conds_t_0_az_d_40_az[i-361]



In [89]:
# all_conds
# all conditions 
with open('target_40_elev_dist_0_elev_and_target_0_elev_dist_40_elev.pkl', 'wb') as f:
    pickle.dump(all_conds, f)

In [91]:
all_conds[721]

[(270, 0), (270, 40)]